# In-class Exercise: Mini Triage Dashboard (Python Basics)

**Context:** You are prototyping backend logic for a simple emergency department triage dashboard. You receive encounter records as a *list of dictionaries* (similar to JSON from an API). Your job is to calculate a triage score, assign a priority bucket, summarize the queue, and choose who should be seen next.

## Learning goals
By completing this notebook you will practice:
- Writing **functions** with parameters and return values
- Using **loops** and **conditionals**
- Working with **lists** and **dictionaries** (including a list of dicts)
- Creating summary outputs useful for dashboards

---

## Instructions
Work top-to-bottom. **Do not skip cells.**
- Cells labeled **TODO** contain places where you must write code.
- Run each cell after you finish it.


## 0) Setup: Provided sample data
Run this cell to load the example encounter data.


In [1]:
encounters = [
    {"id": "E001", "age": 72, "hr": 118, "sbp": 86,  "chief_complaint": "chest pain", "arrival_min": 5},
    {"id": "E002", "age": 34, "hr": 92,  "sbp": 122, "chief_complaint": "headache",   "arrival_min": 12},
    {"id": "E003", "age": 58, "hr": 105, "sbp": 99,  "chief_complaint": "shortness of breath", "arrival_min": 30},
    {"id": "E004", "age": 81, "hr": 88,  "sbp": 140, "chief_complaint": "fall",       "arrival_min": 55},
    {"id": "E005", "age": 47, "hr": 131, "sbp": 92,  "chief_complaint": "fever",      "arrival_min": 18},
    {"id": "E006", "age": 29, "hr": 76,  "sbp": 110, "chief_complaint": "abdominal pain", "arrival_min": 40},
]

high_risk_complaints = ["chest pain", "shortness of breath", "stroke symptoms"]

print("Loaded", len(encounters), "encounters")
print("Example record:", encounters[0])

Loaded 6 encounters
Example record: {'id': 'E001', 'age': 72, 'hr': 118, 'sbp': 86, 'chief_complaint': 'chest pain', 'arrival_min': 5}


# 1) Warm-up: List operations (8–12 minutes)

This warm-up prepares you for the main exercise. You will practice:
- indexing (first/last)
- appending items
- looping through a list
- building a new list

## 1A) Micro-drills with a simple list
Complete the TODOs and run the cell.


In [7]:
# WARM-UP LIST MICRO-DRILLS

temps_f = [98.6, 101.2, 99.1, 102.5, 97.9]

# 1) Indexing: print the first and last items
print("First temp:", temps_f[0])
print("Last temp:", temps_f[-1])

# 2) Append: add a new temperature reading
# TODO: append 100.4 to temps_f
temps_f.append(100.4)


# 3) Loop: count how many temps are >= 100.4 (fever threshold)
fever_count = 0
for t in temps_f:
    if t >=100.4:
            fever_count += 1
    pass

print("Fever count:", fever_count)

# 4) Build a new list: convert all temps to Celsius (rounded to 1 decimal)
temps_c = []
for t in temps_f:
    c = (t - 32) * 5/9
    temps_c.append(round(c,1))
    pass

print("Temps in C:", temps_c)

# 5) Optional: slicing
print("First three readings:", temps_f[:3])

First temp: 98.6
Last temp: 97.9
Fever count: 3
Temps in C: [37.0, 38.4, 37.3, 39.2, 36.6, 38.0]
First three readings: [98.6, 101.2, 99.1]


## 1B) Warm-up with a list of dictionaries
This mirrors the structure of the real encounter data.
Complete the TODOs and run the cell.


In [8]:
# WARM-UP: LIST OF DICTS

mini_encounters = [
    {"id": "E001", "age": 72},
    {"id": "E002", "age": 34},
    {"id": "E003", "age": 58},
]

# 1) Access: print the id of the first encounter
print("First id:", mini_encounters[0]["id"])

# 2) Loop: build a list of just the ids
ids = []
for e in mini_encounters:
    ids.append(e["id"])
    pass

print("IDs:", ids)

# 3) Add a new dict: append a new encounter
# TODO: append {"id":"E004","age":81} to mini_encounters

print("Total encounters now:", len(mini_encounters))
print("Last encounter:", mini_encounters[-1])

First id: E001
IDs: ['E001', 'E002', 'E003']
Total encounters now: 3
Last encounter: {'id': 'E003', 'age': 58}


# 2) Main Exercise: Mini Triage Dashboard (45–60 minutes)

## Overview
You will implement five functions:
1. `triage_score(encounter, high_risk_list)`
2. `priority_bucket(score)`
3. `annotate_encounters(encounters, high_risk_list)`
4. `priority_summary(annotated)`
5. `next_up(annotated)`

### Triage scoring rules
Start score at 0 and add points:
- If `age >= 65`: +2
- If `hr >= 120`: +2
- If `sbp < 90`: +3
- If `chief_complaint` is in `high_risk_list`: +2
- If `arrival_min <= 10`: +1

### Priority buckets
- score **>= 7** → `"RED"`
- score **4–6** → `"YELLOW"`
- score **0–3** → `"GREEN"`


## 2A) Implement `triage_score(encounter, high_risk_list)`
Complete the function below according to the rules above.

**Tips:**
- Use `encounter["age"]` to access values.
- Use `if` statements to add to `score`.


In [10]:
def triage_score(encounter, high_risk_list):
    score = 0    
    if encounter["age"] >= 65:
        score += 2
        
    if encounter["hr"] >= 120:
        score += 2
        
    if encounter["sbp"] < 90:
        score += 3
        
    if encounter["chief_complaint"] in high_risk_list:
        score += 2
        
    if encounter["arrival_min"] <= 10:
        score += 1
        
    return score
# Quick self-check (should print <class 'int'>)
print(type(triage_score(encounters[0], high_risk_complaints)))

<class 'int'>


## 2B) Implement `priority_bucket(score)`
Return the correct bucket name as a string.


In [11]:
def priority_bucket(score):
    if score >= 7:
        return "RED"
    elif 4 <= score <= 6:
        return "YELLOW"
    else:
        return "GREEN"


# Quick self-check (should print: GREEN YELLOW RED)
print(priority_bucket(0), priority_bucket(4), priority_bucket(7))


GREEN YELLOW RED


## 2C) Implement `annotate_encounters(encounters, high_risk_list)`
Create and return a **new list** where each encounter dict has two additional keys:
- `"score"`
- `"priority"`

**Important:** Do not modify the original dicts in the original list.

**Hint:** You can copy a dictionary using `new_dict = old_dict.copy()`.


In [12]:
def annotate_encounters(encounters, high_risk_list):
    annotated = []
    
    for encounter in encounters:
        new_dict = encounter.copy()   # copy original dict
        
        score = triage_score(new_dict, high_risk_list)
        new_dict["score"] = score
        new_dict["priority"] = priority_bucket(score)
        
        annotated.append(new_dict)
    
    return annotated


# Test
annotated = annotate_encounters(encounters, high_risk_complaints)

print("Original has score key?", "score" in encounters[0])  # Should be False
print("Annotated example:", annotated[0] if annotated else "(empty)")


Original has score key? False
Annotated example: {'id': 'E001', 'age': 72, 'hr': 118, 'sbp': 86, 'chief_complaint': 'chest pain', 'arrival_min': 5, 'score': 8, 'priority': 'RED'}


## 2D) Implement `priority_summary(annotated)`
Return a dictionary of counts by priority.

Example output:
```python
{"RED": 2, "YELLOW": 2, "GREEN": 2}
```

**Requirement:** Ensure all three keys exist even if a count is 0.


In [13]:
def priority_summary(annotated):
    summary = {"RED": 0, "YELLOW": 0, "GREEN": 0}
    
    for encounter in annotated:
        priority = encounter.get("priority")
        if priority in summary:
            summary[priority] += 1
    
    return summary


print("Summary:", priority_summary(annotated))


Summary: {'RED': 1, 'YELLOW': 0, 'GREEN': 5}


## 2E) Implement `next_up(annotated)`
Return the **encounter id** of who should be seen next.

Rules:
- Highest **score** wins
- If there is a tie, the smaller `arrival_min` wins (arrived earlier)

**Hint:** Track the "best" encounter while looping.


In [26]:
def next_up(annotated):
    if not annotated:
        return None
    
    best = annotated[0]
    
    for encounter in annotated[1:]:
        if encounter["score"] > best["score"]:
            best = encounter
        elif encounter["score"] == best["score"]:
            if encounter["arrival_min"] < best["arrival_min"]:
                best = encounter
    
    return best["id"]


print("Next up:", next_up(annotated))


Next up: E001


# 3) Run the dashboard output
When everything is implemented, run this cell.


In [24]:
annotated = annotate_encounters(encounters, high_risk_complaints)

print("Annotated encounters:")
for e in annotated:
    print(e["id"], e["score"], e["priority"], "arrival_min=", e["arrival_min"])

print("\nSummary:", priority_summary(annotated))
print("Next up:", next_up(annotated))


Annotated encounters:
E001 8 RED arrival_min= 5
E002 0 GREEN arrival_min= 12
E003 2 GREEN arrival_min= 30
E004 2 GREEN arrival_min= 55
E005 2 GREEN arrival_min= 18
E006 0 GREEN arrival_min= 40

Summary: {'RED': 1, 'YELLOW': 0, 'GREEN': 5}
Next up: E001


# 4) Stretch (optional)
Pick ONE if you finish early.

## Stretch A: Group encounter IDs by priority
Write a function `group_by_priority(annotated)` that returns:
```python
{"RED": ["E001", ...], "YELLOW": [...], "GREEN": [...]}
```

## Stretch B: Data quality checks
Write `validate(encounter)` that returns `True/False` if required keys exist and values are sensible (e.g., SBP > 0). Skip invalid encounters.

## Stretch C: Make weights configurable
Create a dict of weights and use it in `triage_score` so you can change scoring without rewriting logic.


In [25]:
def group_by_priority(annotated):
    grouped = {"RED": [], "YELLOW": [], "GREEN": []}
    
    for encounter in annotated:
        priority = encounter.get("priority")
        if priority in grouped:
            grouped[priority].append(encounter["id"])
    
    return grouped


# Example usage
print(group_by_priority(annotated))


{'RED': ['E001'], 'YELLOW': [], 'GREEN': ['E002', 'E003', 'E004', 'E005', 'E006']}
